# Ⅰ 第1〜4回 補充課題：自分の写真で①〜⑫を動かし，ビデオで説明する

**対象**：第1〜4回に未提出の回がある人（Open-LMS で案内された人）．**1 人で取り組む**．

**問い**：自分の写真を見分ける MLP をつくり，学習の様子・重みの保存・シードによるばらつきを，自分の結果で説明できるか．

## 進め方

1. 写真を用意する：2 クラス × 各 50 枚以上を `assets/myphotos/<クラス名>/` に入れ，ターミナルで `python setup/prepare_mydata.py`（第4回と同じ．フォルダ名は半角英数字．顔・個人情報は写さない）
2. このノートブックを上から最後まで実行する（①〜⑫は書いてある．読んで意味を確かめる）
3. ビデオを撮る（下の「ビデオの撮り方」）
4. 最後のセルに 4 つの問いの答えを書き，保存する

## ビデオの撮り方

- 画面録画に **自分の声** を入れる．顔は写さなくてよい．**5 分以内・1 本撮り**（つなぐ編集はしない）
- Mac は `Shift + Command + 5` →「画面全体を収録」→ オプションでマイクを選ぶ
- 撮ったファイルを QuickTime Player で開き，「ファイル」→「書き出す」→「720p」で保存する（Open-LMS は 1 ファイル 50 MB まで．超えたら「480p」）
- ビデオの中で次の順に行う
  1. ⑪のグラフと⑫の正解率を見せ，下の **問い 1〜4** に答える．数値を言うときは画面のその数値を **カーソルで指す**
  2. **(0) で出た「変える所」** を言い，⑦〜⑫を実行し直す **前に**，検証損失のグラフとテスト正解率がどう変わるかを **予測して言う**
  3. その 1 行を変えて⑦〜⑫を実行し直し，結果を見て，予測と比べて当たった・外れた理由を言う

| 学籍番号の下 2 桁 ÷ 3 の余り | 変える所 | 変え方 |
|---|---|---|
| 0 | ⑨ | 学習率 `lr=0.001` を `lr=0.01` にする |
| 1 | ⑦ | `HIDDEN_SIZE1 = 512` を `32` に，`HIDDEN_SIZE2 = 256` を `16` にする |
| 2 | ④ | `BATCH_SIZE = 8` を `64` にする |

## 問い（ビデオで答え，最後のセルにも書く）

1. （第1回）⑦の MLP の各行は何をしているか．`INPUT_SIZE` の値はいくつで，どこから来た数か
2. （第2回）保存した重みを読むとき，なぜ⑦をもう一度実行してから `load_state_dict` するのか
3. （第3回）自分のグラフで，過学習はどこから始まったか．どの線を見てそう言えるか
4. （第4回）自分の「差 ÷ 標準偏差」の値と判定．テストの枚数が増えると，判定はどう変わりうるか

## 提出（期限：11/12（木）13:10）

Open-LMS の「第1〜4回 補充課題」へ：このノートブック `makeup.ipynb` と，書き出したビデオのファイル（50 MB まで）．写真と `makeup_mlp.pt` は出さない．

In [ ]:
# ===== (0) 設定 =====
STUDENT_ID = ""               # ← 自分の学籍番号（7 桁）を書く

import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

CONDITIONS = {0: ("⑨", "学習率 lr=0.001 を lr=0.01 にする"), 1: ("⑦", "HIDDEN_SIZE1 = 512 を 32 に，HIDDEN_SIZE2 = 256 を 16 にする"), 2: ("④", "BATCH_SIZE = 8 を 64 にする")}
if not (STUDENT_ID.isdigit() and len(STUDENT_ID) == 7):
    raise RuntimeError("STUDENT_ID に自分の学籍番号（7 桁の数字）を書く")
step, change = CONDITIONS[int(STUDENT_ID[-2:]) % 3]
print(f"あなたがビデオの中で変える所：{step}　{change}")

## (1) 自分のデータ

In [ ]:
from common.mydata import MYDATA
if not (MYDATA.exists() and len([d for d in MYDATA.iterdir() if d.is_dir()]) >= 2):
    raise RuntimeError("assets/mydata/ がない．写真を assets/myphotos/<クラス名>/ に入れ，"
                       "ターミナルで python setup/prepare_mydata.py を実行する")
IMAGE_DIR = str(MYDATA)
for d in sorted(MYDATA.iterdir()):
    if d.is_dir():
        print(f"{d.name:20s} {len(list(d.glob('*.jpg')))} 枚")

## 第1回：①〜⑫で MLP をつくる（50 エポック）

### コードテンプレート①：ライブラリのインポート

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from torch.utils.data import Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import random
import numpy as np
from torchvision.datasets import ImageFolder

### コードテンプレート②：データ前処理

In [ ]:
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ]),
    'val': transforms.Compose([
        transforms.Resize((64, 64)),
        transforms.ToTensor(),
    ])
}

### コードテンプレート③：データの読み込みと分割（乱数の種の固定）

In [ ]:
full_dataset = ImageFolder(root=IMAGE_DIR)
random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(0)
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_subset, val_subset, test_subset = random_split(full_dataset, [train_size, val_size, test_size])
class CustomDataset(torch.utils.data.Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __getitem__(self, index):
        image, label = self.subset[index]
        if self.transform:
            image = self.transform(image)
        return image, label
    def __len__(self):
        return len(self.subset)
train_dataset = CustomDataset(train_subset, transform=data_transforms['train'])
val_dataset = CustomDataset(val_subset, transform=data_transforms['val'])
test_dataset = CustomDataset(test_subset, transform=data_transforms['val'])

### コードテンプレート④：データローダの作成

In [ ]:
BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

### コードテンプレート⑤：使用デバイスの選択

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"使用デバイス: {device}")

### コードテンプレート⑥：ミニバッチの形状確認

In [ ]:
imgs, _ = next(iter(train_loader))
c, h, w = imgs[0].shape
print(imgs.shape)

### コードテンプレート⑦：ネットワークの定義（MLP）

In [ ]:
INPUT_SIZE = c * h * w
HIDDEN_SIZE1 = 512
HIDDEN_SIZE2 = 256
OUTPUT_SIZE = len(full_dataset.classes)
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(INPUT_SIZE, HIDDEN_SIZE1),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE1, HIDDEN_SIZE2),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE2, OUTPUT_SIZE)
)
model.to(device)

### コードテンプレート⑧⑨：損失関数と最適化手法

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### コードテンプレート⑩：学習（50 エポック）

In [ ]:
EPOCHS = 50
train_loss_list = []
val_loss_list = []
train_acc_list = []
val_acc_list = []
import time
_t0 = time.perf_counter()
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    train_loss_list.append(train_loss / train_total)
    train_acc_list.append(100.0 * train_correct / train_total)
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    val_loss_list.append(val_loss / val_total)
    val_acc_list.append(100.0 * val_correct / val_total)
train_sec = time.perf_counter() - _t0

### コードテンプレート⑪：学習結果の可視化（左：損失，右：正解率）

In [ ]:
epochs = range(1, len(train_loss_list) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(x=epochs, y=train_loss_list, label='Train Loss', ax=axes[0])
sns.lineplot(x=epochs, y=val_loss_list, label='Val Loss', ax=axes[0])
axes[0].set_xlabel('Epoch')
sns.lineplot(x=epochs, y=train_acc_list, label='Train Accuracy', ax=axes[1])
sns.lineplot(x=epochs, y=val_acc_list, label='Val Accuracy', ax=axes[1])
axes[1].set_xlabel('Epoch')
plt.show()
best = min(range(len(val_loss_list)), key=lambda i: val_loss_list[i]) + 1
print(f"検証損失が一番小さいのは {best} エポック目（{min(val_loss_list):.3f}）．50 エポック目は {val_loss_list[-1]:.3f}")

### コードテンプレート⑫：テストデータでの評価

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {correct / total * 100:.2f}% ')

## 第2回：重みを保存し，組み立て直して読む

保存して，いまの `model` を消す．

In [ ]:
SAVE_PATH = ROOT / "results" / "makeup_mlp.pt"
torch.save(model.state_dict(), SAVE_PATH)     # 重みの数値（state_dict）だけを保存する
print("保存した:", SAVE_PATH)
del model                                     # いまのモデルを消す（別のノートブックで開いた状態にする）

⑦をもう一度実行して，同じ構造を組み立てる（下のセルは⑦と同じ）．

In [ ]:
INPUT_SIZE = c * h * w
HIDDEN_SIZE1 = 512
HIDDEN_SIZE2 = 256
OUTPUT_SIZE = len(full_dataset.classes)
model = nn.Sequential(
    nn.Flatten(),
    nn.Linear(INPUT_SIZE, HIDDEN_SIZE1),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE1, HIDDEN_SIZE2),
    nn.ReLU(),
    nn.Linear(HIDDEN_SIZE2, OUTPUT_SIZE)
)
model.to(device)

保存した重みを入れ，⑫と同じ方法でテストする．上の⑫と同じ正解率になるはず．

In [ ]:
model.load_state_dict(torch.load(SAVE_PATH))  # 上で組み立てた model に，保存した重みを入れる
model.eval()

In [ ]:
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
print(f'Accuracy: {correct / total * 100:.2f}% ')

## 第4回：MLP と CNN をシード 0〜4 で回し，差 ÷ 標準偏差 で判定する（20 エポック・1 回数秒）

In [ ]:
import pandas as pd
from common.mydata import run, diff_ratio, verdict
rows = []
for kind in ["MLP", "CNN"]:
    for seed in range(5):
        r = run(kind, seed=seed, device=device, root=MYDATA)
        rows.append({"モデル": kind, "シード": seed, "テスト正解率": 100 * r["test_acc"]})
        print(f"{kind} シード {seed}：テスト正解率 {100 * r['test_acc']:.1f}%")
print(f"テストは {r['n_test']} 枚．1 枚で {100 / r['n_test']:.1f} ポイント動く")
df = pd.DataFrame(rows)
print(df.groupby("モデル")["テスト正解率"].agg(平均="mean", 標準偏差="std").round(1).to_string())
diff, sd, ratio = diff_ratio(df[df.モデル == "MLP"].テスト正解率, df[df.モデル == "CNN"].テスト正解率)
print(f"平均の差（CNN − MLP）{diff:+.1f} ポイント ÷ 標準偏差 {sd:.1f} = {ratio:.1f} → {verdict(ratio)}")

## ビデオの後半：(0) で出た所を 1 行変えて，⑦〜⑫を実行し直す

予測を声で言ってから，上の該当セルを書き換え，⑦ → ⑧⑨ → ⑩ → ⑪ → ⑫ の順に実行する（④を変える人は④から）．

## 答え（提出）

このセルをダブルクリックして，＿＿ に書き込む（それぞれ 1〜2 文．ビデオで話したことと同じでよい）．

1. ⑦の各行の役割と，`INPUT_SIZE` の値・由来：＿＿
2. ⑦をもう一度実行してから `load_state_dict` する理由：＿＿
3. 過学習が始まったエポックと，そう言える根拠：＿＿
4. 差 ÷ 標準偏差 の値と判定．テストの枚数が増えたらどうなりうるか：＿＿
5. 変えた所と予測．結果と比べて当たった・外れた理由：＿＿